# 06 — Aggregator — pool the statistics, build the catalogue

| | |
|---|---|
| **runs at** | the aggregator |
| **reads** | `cohort-A_stats.npz`, `cohort-B_stats.npz`, `cohort-A_labels.txt` |
| **writes** | `federated_distance.csv`, `federated_pvclust_catalogue.csv` |

The aggregator now holds two sets of four matrices. It sums them.

**Why this is exact, not approximate.** Every entry of every matrix is a sum over
rows. Summing the matrices from two cohorts therefore gives precisely the matrices
you would have computed from the two cohorts' rows side by side. The pooled
distance is **bit-for-bit** the distance from pooled raw data — verified to machine
precision in the test suite — and so is the tree built from it.

That is the whole claim of this design: no subject-level row moved, and nothing
was lost by not moving it.

In [ ]:
import os, subprocess
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Every notebook reads and writes here, so the chain hands work along. In a real
# deployment this directory would be on a different machine for each cohort, and
# only the files named 'what this writes' would ever cross the boundary.
WORK = Path('sle-run'); WORK.mkdir(exist_ok=True); os.chdir(WORK)

def run(cmd):
    """Run one pvclust-py command and echo it, so the notebook shows the command
    line rather than hiding it behind a function."""
    print('$ ' + ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1800:])
    if r.returncode:
        raise SystemExit(f'command failed: {cmd}')

def show(png):
    """Render a written figure inline. matplotlib only, so this works in CI too."""
    if not Path(png).exists():
        print(f'(missing {png})'); return
    fig, ax = plt.subplots(figsize=(13, 13))
    ax.imshow(plt.imread(png)); ax.axis('off'); ax.set_title(png, fontsize=9)
    plt.show()

In [ ]:
run(['pvclust-py', 'aggregate-trees',
     '--labels', 'cohort-A_labels.txt',
     '--stats', 'cohort-A_stats.npz',
     '--stats', 'cohort-B_stats.npz',
     '--dist', 'minkowski', '--linkage', 'ward.D2'])

## The catalogue

The clusters of the pooled tree become the **catalogue**: the fixed list of
clusters that every cohort will count against in the next step.

Each carries an `edge_id`, a hash of its sorted member names. Identity by content,
never by tree position — that is what lets two cohorts agree they are counting the
same thing.

In [ ]:
cat = pd.read_csv('federated_pvclust_catalogue.csv')
print(f'{len(cat)} clusters in the catalogue')
print()
small = cat[cat.n_members.between(2, 6)].head(6)
for _, r in small.iterrows():
    print(f"  {r.edge_id}  n={r.n_members}  {r.members[:66]}")

## Sanity check on the pooled distance

The pooled matrix should be symmetric, have a zero diagonal, and contain no
non-finite values. A non-finite entry means some pair of proteins had no samples
in common, or one of them was constant — the kind of thing step 01 checked for.

In [ ]:
D = pd.read_csv('federated_distance.csv', index_col=0)
M = D.to_numpy(float)
print(f'shape           : {M.shape}')
print(f'symmetric       : {np.allclose(M, M.T)}')
print(f'zero diagonal   : {np.allclose(np.diag(M), 0)}')
print(f'all finite      : {np.isfinite(M).all()}')
print(f'range           : {M[M>0].min():.4f} to {M.max():.4f}')

---
**Why there must now be a second pass.** The catalogue exists, but no *p*-value
does. Each cohort's own run in step 04/05 counted the clusters of **its own** tree,
and those differ between cohorts, so the counts are not comparable and cannot be
summed.

This is not a technicality. Pooling per-cohort trees instead finds clusters that
no cohort actually agrees on.

**Next:** `07` and `08`. Every cohort counts this one catalogue.